# Figure 3 Reproduction — Massive Activations in FLUX.2-klein (Colab runner)

End-to-end Colab runner for **Figure 3 / Section 3.2** of *"Few Channels Draw The Whole
Picture: Revealing Massive Activations in Diffusion Transformers"* (arXiv:2605.13974),
reproduced for **FLUX.2-klein**. See `README.md` / `SPEC.md` in the repo for the full
pipeline design, invariants, and acceptance criteria — this notebook just drives it.

**Before you run:**
1. `Runtime > Change runtime type` → GPU (T4 works; A100/L4 is faster and supports `bf16`).
2. You likely need a Hugging Face account with access to `black-forest-labs/FLUX.2-klein`
   and `ZhengPeng7/BiRefNet`, plus an access token (Colab left sidebar → 🔑 *Secrets* →
   add `HF_TOKEN`; otherwise you'll be prompted to log in interactively).
3. The full run needs the 1,600 GenAI-Bench prompts (`.txt`/`.json`/`.jsonl`/`.parquet`,
   or an HF dataset id) — set `PROMPT_SOURCE` in the config cell. A tiny built-in prompt
   list is used for the smoke test first so you can validate everything end to end quickly.
4. Mount Google Drive (below) so outputs/cache survive a session disconnect — the
   pipeline is resumable either way (stage1 skips already-cached prompts on restart).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader \
    || echo "No GPU detected — set Runtime > Change runtime type > GPU"
import sys

print(sys.version)

## 1. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/BrendanGho/massive-activations-fig3.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
REPO_DIR = "/content/massive-activations-fig3"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists — pulling latest {BRANCH}...")
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}

## 2. (Optional) Mount Google Drive for persistent storage

Recommended for the full 1,600-prompt run — writes then survive a session ending
mid-run. Skip this and outputs just live in the (ephemeral) Colab VM instead.

In [ ]:
USE_DRIVE = True  # @param {type:"boolean"}

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/figure3_repro"
else:
    DRIVE_ROOT = "/content/figure3_repro"

OUTPUT_DIR = f"{DRIVE_ROOT}/outputs"
CACHE_DIR = f"{DRIVE_ROOT}/cache"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print("output_dir:", OUTPUT_DIR)
print("activation_cache_dir:", CACHE_DIR)

## 3. Install dependencies

Editable install of this repo plus the `fig3` extra (torch / diffusers / transformers /
scikit-learn / matplotlib / ...). Takes a few minutes on a fresh Colab runtime.

In [ ]:
!pip install -q -e ".[fig3]"

In [ ]:
import torch

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    major, _minor = torch.cuda.get_device_capability(0)
    DEVICE = "cuda"
    # bf16 needs Ampere+ (compute capability >= 8, e.g. A100/L4); T4 (cc 7.5) falls back to fp16.
    DTYPE = "bf16" if major >= 8 else "fp16"
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        "| compute capability:",
        major,
        "| using dtype:",
        DTYPE,
    )
else:
    DEVICE = "cpu"
    DTYPE = "fp32"
    print("No GPU — falling back to CPU/fp32. This will be extremely slow; use a GPU runtime.")

## 4. Hugging Face authentication

`FLUX.2-klein` and/or `BiRefNet` may be gated. Add an `HF_TOKEN` under Colab's 🔑 *Secrets*
panel (recommended, persists across sessions), or log in interactively below.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token)
    print("Logged in via Colab secret HF_TOKEN.")
else:
    login()  # interactive prompt

## 5. Configure the run

Fills the config loader's required keys (`src/common/config.py`) and writes a
Colab-specific `configs/colab.yaml` — the committed `configs/default.yaml` stays untouched.
`PROMPT_SOURCE` is the one value you should repoint at real data before the full run
(section 7); it's left as the smoke-test file until then.

In [ ]:
import yaml

MODEL_CKPT = "black-forest-labs/FLUX.2-klein"  # @param {type:"string"}
BIREFNET_WEIGHTS = "ZhengPeng7/BiRefNet"  # @param {type:"string"}
# 1,600 GenAI-Bench prompts: point this at a .txt/.json/.jsonl/.parquet file or an HF
# dataset id before the full run (section 7). Left as the smoke-test file until then.
PROMPT_SOURCE = f"{OUTPUT_DIR}/smoke_test_prompts.txt"  # @param {type:"string"}

# A handful of prompts so stage1/stage4 can be sanity-checked before the full run.
_SMOKE_PROMPTS = [
    "a red bicycle leaning against a brick wall",
    "a golden retriever sitting on a wooden dock at sunset",
    "a bowl of fresh strawberries on a marble countertop",
    "an astronaut planting a flag on the moon",
]
if not os.path.isfile(PROMPT_SOURCE):
    with open(PROMPT_SOURCE, "w") as fh:
        fh.write("\n".join(_SMOKE_PROMPTS) + "\n")

config = {
    "model_ckpt": MODEL_CKPT,
    "prompt_source": PROMPT_SOURCE,
    "birefnet_weights": BIREFNET_WEIGHTS,
    "output_dir": OUTPUT_DIR,
    "activation_cache_dir": CACHE_DIR,
    "num_denoising_steps": 4,
    "resolution": 1024,
    "top_k": 12,
    "layers": "all",
    "random_k_trials": 5,
    "seed": 0,
    "device": DEVICE,
    "dtype": DTYPE,
    "batch_size": 1,
    "num_example_prompts": 8,
    "cache_batch_size": 32,
    "guidance_scale": None,
}

CONFIG_PATH = "configs/colab.yaml"
with open(CONFIG_PATH, "w") as fh:
    yaml.safe_dump(config, fh, sort_keys=False)

print(open(CONFIG_PATH).read())

## 6. Quick smoke test

Runs the 4 built-in prompts through both stages end to end — catches config, auth, or
model-loading issues in minutes instead of partway through the full 1,600-prompt run.

In [ ]:
!python -m src.stage1_generate_and_cache --config {CONFIG_PATH} --fused --limit 4
!python -m src.stage4_evaluate_figure3d --config {CONFIG_PATH}

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f"{OUTPUT_DIR}/figure3d_results.csv")
display(df.pivot(index="layer", columns="strategy", values="mean_miou"))
display(Image(filename=f"{OUTPUT_DIR}/figure3d_curve.png"))

## 7. Full run (1,600 GenAI-Bench prompts)

Point `PROMPT_SOURCE` in section 5 at the real prompt set, re-run that cell to regenerate
`configs/colab.yaml`, then run the full resumable pipeline below. This is the expensive
step — budget real GPU time. If the session disconnects mid-run, just re-run this cell:
`stage1` skips prompts already in `activation_cache_dir`/`completed.json`.

In [ ]:
!bash scripts/run_pipeline.sh {CONFIG_PATH}

## 8. Results

In [ ]:
import glob

import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f"{OUTPUT_DIR}/figure3d_results.csv")
display(df.pivot(index="layer", columns="strategy", values="mean_miou"))
display(Image(filename=f"{OUTPUT_DIR}/figure3d_curve.png"))

for qdir in sorted(glob.glob(f"{OUTPUT_DIR}/qualitative/prompt_*"))[:3]:
    print(qdir)
    for png in sorted(glob.glob(f"{qdir}/*.png"))[:4]:
        display(Image(filename=png, width=220))

## 9. (Optional) Download outputs

Only needed if you didn't mount Drive in section 2 — otherwise outputs already live in
your Drive at `OUTPUT_DIR`.

In [ ]:
if not USE_DRIVE:
    import shutil
    from google.colab import files

    archive = shutil.make_archive("/content/figure3_outputs", "zip", OUTPUT_DIR)
    files.download(archive)
else:
    print("Outputs are already in Google Drive:", OUTPUT_DIR)